# Second Model: Forecasting GMM

A fundamental limitation of a GMM is that it's a density estimation model. It learns $p(\textbf{x})$ and has no concept of time evolution $p(\textbf{x}_{t+1}|\textbf{x}_t)$ at time-step $t$, like a Hidden Markov Model. Our previous model is good at evaluating whether a current vector of environmental factors and vegetation indices is likely to occur given the ones already observed, but unfortunately it cannot be used in its original form as a forecasting model. We now consider a proposal that the GMM can be modified to allow for forecasting by making use of the EM-algorithm.

Eirola and Lendasse (2013) proposed that, making use of **delay embedding** of length $d$ to create $d$-dimensional overlapping rolling windows from a time series, we can fit a GMM to the resulting vectors and forecast future observations using the conditional expectation of a multivariate Gaussian. This allows for the prediction of an entire future horizon simultaneously.

The paper specifically mentions that this method does not perform well with high dimensional data due to significant overfitting. However, as a proof of concept for this project and a generative baseline, this notebook is an implementation of the second section of this paper, with plans to create the eventual constrained model in section 3.

## Libraries

In [378]:
import sys
from pathlib import Path
import ee

sys.path.append(str(Path.cwd().parent / "src"))
print("Authenticating with Google Earth Engine...")
ee.Authenticate()
print("Initializing project...")
ee.Initialize(project="agriculture-drought-assesment")

Authenticating with Google Earth Engine...
Initializing project...


In [379]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", context="notebook", palette="deep")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.mixture import GaussianMixture 
from sklearn.decomposition import PCA

from scipy.stats import multivariate_normal
from scipy.special import logsumexp

from data import utils

## Functions

In [406]:
# Extract extreme drought years 2018, 2019

def get_anomaly_df(df):
    # df = rem_year(df, 2025)
    mask = (df["Ag_year"].isin([]))
    return df[~mask].copy(), df[mask].copy()

def rem_year(df, year, *years):
    all_years = [year, *years]
    mask = df["Timestamp"].dt.year.isin(all_years)
    return df[~mask]

def get_interpolated_df(spectral, timestamps):
    spectral_columns = [
        "NDVI_mean",
        "NDWI_mean",
        "NDRE_mean",
    ]

    df = spectral.merge(timestamps, on="Timestamp", how="right")
    interpolated_df = df[["Timestamp"] + spectral_columns].copy()
    interpolated_df[spectral_columns] = interpolated_df[spectral_columns].interpolate(method="pchip", limit_area="inside").bfill()

    return interpolated_df.dropna()

## Data

In [407]:
non_spectral_df = pd.read_csv("../data/raw/non_spectral_df.csv", index_col=0)
non_spectral_df["Timestamp"] = pd.to_datetime(non_spectral_df["Timestamp"])

spectral_df = pd.read_csv("../data/raw/spectral_df.csv", index_col=0)
spectral_df["Timestamp"] = pd.to_datetime(spectral_df["Timestamp"])

grouped_spectral_df = pd.read_csv("../data/raw/grouped_spectral_df.csv", index_col=0)
grouped_spectral_df["Timestamp"] = pd.to_datetime(grouped_spectral_df["Timestamp"])

In [408]:
eng_non_spectral_df  = pd.read_csv("../data/processed/eng_non_spectral_df.csv", index_col=0)
eng_non_spectral_df["Timestamp"] = pd.to_datetime(eng_non_spectral_df["Timestamp"])
eng_non_spectral_df["in_season"] = eng_non_spectral_df["in_season"].astype('category')
eng_non_spectral_df["Growth_Stage"] = eng_non_spectral_df["Growth_Stage"].astype('category')

In [409]:
# Rolling statistics and lagged variables
roll_lag_df = pd.read_csv("../data/processed/roll_lag_df.csv", index_col=0)
roll_lag_df["Timestamp"] = pd.to_datetime(roll_lag_df["Timestamp"])

In [410]:
non_spectral_df = non_spectral_df.merge(
    eng_non_spectral_df[["Timestamp", "Ag_year", "shifted_doy"]],
    on="Timestamp",
    how="left"
)

grouped_spectral_df = grouped_spectral_df.merge(
    eng_non_spectral_df[["Timestamp", "Ag_year", "shifted_doy"]],
    on="Timestamp",
    how="left"
)

### Get Anomaly Data

In [411]:
norm_ns_df, anom_ns_df = get_anomaly_df(non_spectral_df)
norm_s_df, anom_s_df = get_anomaly_df(grouped_spectral_df[["Timestamp", "Ag_year", "NDVI_mean", "NDWI_mean", "NDRE_mean"]])

norm_eng_ns_df, anom_eng_ns_df = get_anomaly_df(eng_non_spectral_df)
norm_roll_lag_df, anom_roll_lag_df = get_anomaly_df(roll_lag_df)

### Interpolation

The paper by Eirola and Lendasse (n.d.) makes use of a modified EM algorithm that allows for the imputation of missing time series values under the assumption that the observations are missing at random (MAR). We will not use this since vegetation indices follow quite a predictable pattern that can mostly be captured by interpolation (as seen in the previous GMM).

In [416]:
norm_interpolated_df = get_interpolated_df(spectral=norm_s_df, timestamps=norm_eng_ns_df["Timestamp"])
anom_interpolated_df = get_interpolated_df(spectral=anom_s_df, timestamps=anom_eng_ns_df["Timestamp"])

---

## Feature Arrays

In [417]:
eng_features = [
    # Timestamp for merging
    'Timestamp',

    # Date features for plotting
    "shifted_doy",
    "Ag_year",

    # engineered features
    'cumulative_GDD', 
    'root_weighted_soil_moisture', 
    #'VW_PC1',  
    #'Shallow_mean',
    'ST_PC1',

    # Categoricals not included
    #'Growth_Stage',
    'in_season',
]

non_spectral_features = [
    'Timestamp',
    'temperature_2m',
    #'temperature_2m_min',
    #'temperature_2m_max', 
    #'surface_solar_radiation_downwards_sum',
    'precipitation',
    'volumetric_soil_water_layer_3',
    #'volumetric_soil_water_layer_4',
    'soil_temperature_level_4'
]


## Delay Embedding

As previously stated, a GMM operates on vectors rather than sequential observations. Consequently, we transform the time series into _overlapping_ windows of length $d$. If the univariate time series is given by $$\textbf{z}=[z_0,z_1,...,z_{n-1}]$$
and we choose the dimension of embedding as $d$, then _each training sample_ is constructed as $$\textbf{x}_t=[z_t, z_{t+1}, ..., z_{t+d-1}], \quad t=0,1,...,n-d$$
From this we form the new data matrix $X:(n-d+1)\times d$ where the rows are in $\mathbb{R}^d$. 

An important thing to note is that this is for one time series - we consider several related variables. In other words, we embed a multivariate time series, with $m$ variables per each observation $t=0,1,...,n-1$, such that $X\in\mathbb{R}^{(n-d+1)\times(md)}$

In [418]:
d = 12

In [419]:
def delay_embedding(X, d):
    X = np.asarray(X)

    n, m = X.shape

    embedded = np.empty(shape=(n-d+1, m * d), dtype=X.dtype)

    for i in range(n-d+1):
        embedded[i] = X[i:i+d].reshape(-1)

    return embedded

In [421]:
# Get normal data
X_part1 = norm_eng_ns_df[eng_features]
X_part2 = norm_ns_df[non_spectral_features]
X_non_spectral_norm = pd.merge(X_part2, X_part1, on="Timestamp")

# Merge with interpolated spectral dataframe
X_norm = norm_interpolated_df.merge(X_non_spectral_norm, on="Timestamp", how="left")
X_norm['doy_sin'] = np.sin(2 * np.pi * X_norm['shifted_doy'] / 365.25)
X_norm['doy_cos'] = np.cos(2 * np.pi * X_norm['shifted_doy'] / 365.25)
X_norm["Ag_year"] = X_norm["Ag_year"].astype('int32')
X_norm = X_norm[X_norm["Ag_year"] != 2017]

X_norm.tail()

,Timestamp,NDVI_mean,NDWI_mean,NDRE_mean,temperature_2m,precipitation,volumetric_soil_water_layer_3,soil_temperature_level_4,shifted_doy,Ag_year,cumulative_GDD,root_weighted_soil_moisture,ST_PC1,in_season,doy_sin,doy_cos
2847,2025-10-18,0.359621,0.025982,0.237076,290.307418,7.262418,0.322494,286.845853,357,2024,0.0,0.0,0.242459,0,-0.141444,0.989946
2848,2025-10-19,0.359978,0.021275,0.237818,286.427042,7.953155,0.322192,286.884666,358,2024,0.0,0.0,-0.056132,0,-0.124395,0.992233
2849,2025-10-20,0.360230,0.015715,0.238357,284.402085,0.000000,0.322087,286.922756,359,2024,0.0,0.0,-0.541409,0,-0.107308,0.994226
2850,2025-10-21,0.360379,0.009230,0.238687,286.209297,0.000000,0.321753,286.957585,360,2024,0.0,0.0,-0.353307,0,-0.090190,0.995925
2851,2025-10-22,0.360429,0.001749,0.238799,287.814511,0.000000,0.321080,286.990019,361,2024,0.0,0.0,-0.204881,0,-0.073045,0.997329


In [422]:
# Get anomaly data

X_part1 = anom_eng_ns_df[eng_features]
X_part2 = anom_ns_df[non_spectral_features]
X_ns_anom = pd.merge(X_part2, X_part1, on="Timestamp")

# Merge with interpolated spectral dataframe
X_anom = anom_interpolated_df.merge(X_ns_anom, on="Timestamp", how="left")
X_anom['doy_sin'] = np.sin(2 * np.pi * X_anom['shifted_doy'] / 365.25)
X_anom['doy_cos'] = np.cos(2 * np.pi * X_anom['shifted_doy'] / 365.25)
X_anom["Ag_year"] = X_anom["Ag_year"].astype('int32')
X_anom = X_anom[X_anom["Ag_year"] != 2017]

X_anom.head()

,Timestamp,NDVI_mean,NDWI_mean,NDRE_mean,temperature_2m,precipitation,volumetric_soil_water_layer_3,soil_temperature_level_4,shifted_doy,Ag_year,cumulative_GDD,root_weighted_soil_moisture,ST_PC1,in_season,doy_sin,doy_cos


In [423]:
scaler = StandardScaler()

X = X_norm.drop(["Timestamp", "shifted_doy", "Ag_year", "doy_sin", "doy_cos", "in_season"], axis=1).copy()


test_mask = X_norm["Ag_year"] == 2022
train_mask = ~test_mask

X_train_raw = X[train_mask]
X_test_raw = X[test_mask]

scaler.fit(X_train_raw)

X_train_scaled = scaler.transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

X_train = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train_raw.index)
X_test = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test_raw.index)

X_train_df = pd.concat([X_train, X_norm.loc[train_mask, ["doy_sin", "doy_cos"]]], axis=1)
X_test_df = pd.concat([X_test, X_norm.loc[test_mask, ["doy_sin", "doy_cos"]]], axis=1)

X_train = delay_embedding(X_train_df, d)
X_test = X_test_df.to_numpy()

In [ ]:
Xa = X_anom.drop(["Timestamp", "shifted_doy", "Ag_year", "doy_sin", "doy_cos", "in_season"], axis=1).copy()

X_anom_scaled = scaler.transform(Xa)
X_anom_df = pd.DataFrame(X_anom_scaled, columns=Xa.columns, index=X_anom.index)
X_anom_df = pd.concat([X_anom_df, X_anom[["doy_sin", "doy_cos"]]], axis=1)

X_anom_embedded = delay_embedding(X_anom_df, d)

---

## Conditional Expectations (i.e. Forecasting)

With our regressor size of $d=24$, we can for instance take the last year's measurements as the _first_ 12 months ($P$, known/given), then calculate the conditional expectation of the _next_ 12 months ($F$, unknown). 

$$E[\textbf{x}^F|\textbf{x}^P]$$

Now since each Gaussian component is partitioned into splits $P, F$, it can be shown that for a single component $k$, the conditional expectation of future values conditioned on past sample $\textbf{x}_i^P$ is given by 

$$\tilde{\textbf{y}}_{ik}=E[\textbf{x}^F_i|\textbf{x}^P_i]=\boldsymbol{\mu}_k^F+\Sigma^{FP}_k(\Sigma^{PP}_k)^{-1}(\textbf{x}_i^P-\boldsymbol{\mu}_k^P)$$




Of course, we don't know which component generated point $i$, so if the posterior probability of sample $\textbf{x}_i^P$ belonging to each component $k$ is given by $t_{ik}$ (calculated during the E-step), then the overall prediction is the weighted average 

$$\hat{\textbf{y}}_i=\sum^K_{k=1}t_{ik}\tilde{\textbf{y}}_{ik}$$

### Training

In [428]:
X_test.shape

(365, 12)

In [429]:
covariance_types = ['full']
components = range(1, 10)
bics = np.zeros(shape=(len(covariance_types), len(components)))

min_bic = np.inf
min_bic_k = 0
best_cov = None
best_model = None


for i, cov in enumerate(covariance_types):
    for j, k in enumerate(components):
        gmm = GaussianMixture(n_components=k, covariance_type=cov, random_state=42, n_init=10, reg_covar=1e-3)
        gmm.fit(X_train)
        bic = gmm.aic(X_train)
        bics[i, j] = bic
        if bic < min_bic:
            min_bic = bic
            min_bic_k = k
            best_cov = cov
            best_model = gmm

print(f"Minimum BIC: {min_bic} with {min_bic_k} components and '{best_cov}' Covariance Matrices.")

Minimum BIC: -844653.9164072182 with 4 components and 'full' Covariance Matrices.


In [430]:
n_components = 4
gmm = GaussianMixture(n_components=n_components, covariance_type='full', n_init=10, reg_covar=1e-4)
gmm.fit(X_train)

props = gmm.weights_
means = gmm.means_
covs = gmm.covariances_
covs.shape

(4, 144, 144)

### Prediction

Pretend the last 12 observations of `X_test` are unknown and are to be predicted. Since we trained the GMM on $d=24$, the past $12$ observations are known (denoted by $P$) and the future 12 are (assumed) unknown (denoted by $F$). We form a vector `pred_window` that represents a single window, partitioned into the last 12 known observations and the future 12 set as NaN.

In [432]:
h = d//2 # forecast

true_vals = X_test[-h:] # strictly used for eval - assumed unknown

test_obs = X_test[-2*h : -h] # known past time series
temp = np.full((h, X_test.shape[1]), np.nan)
pred_window = np.vstack([test_obs, temp])

# GMM was trained on n x 288 data so flatten pred window to match
pred_window = pred_window.reshape(1, -1)
pred_window.shape

(1, 144)

Now since the original paper is restricted to a univariate case, we take some creative liberty in the extension to multivariate. In the above `pred_window`, $P\in\mathbb{R}^{144}$ (12 observations and 12 features). A natural extension would be to let $F\in\mathbb{R}^{144}$ and predict the values of each feature. This is unnecessary, as we are not interested in (nor capable of)  accurately forecasting the solar radiation or the precipitation. What would be beneficial however is forecasting the vegetation indices $\text{NDVI}, \ \text{NDWI}, \text{ and } \text{NDRE}$, as these give an interpretable and reliable representation of water and vegatative stress. This means that we will have 3 features and 12 values, so $F\in\mathbb{R}^{36}$.

In [433]:
spectral_indices = ["NDVI_mean", "NDRE_mean", "NDWI_mean"]
responses = np.array([X.columns.get_loc(s) for s in spectral_indices])

In [434]:
X_test.shape[1]

12

In [435]:
future_mask = np.full(shape=(d, X_test.shape[1]), fill_value=False, dtype=bool)
future_mask[h:, responses] = True
future_mask = future_mask.flatten()

past_mask = np.full(shape=(d, X_test.shape[1]), fill_value=False, dtype=bool)
past_mask[:h, :] = True
past_mask = past_mask.flatten()

x_P = pred_window[0, past_mask]

Next we iterate through each Gaussian component, partition according to past/future, calculate the expectations and then the weighted average. Note we can't use `gmm.weights_` for $t_{ik}$ since that represents the global probability of each component across the entire dataset (priors), while $t_{ik}$ represents the specific probability of a component _given_ the past data (posterior probabilities). 

In [436]:
K = n_components
t_ik = np.empty(shape=(K))
y_ik = np.empty((K, future_mask.sum()))

# log contributions for probabilities in log space
log_contribs = np.empty(shape=K)


In [437]:
# calculate each t_ik

for k in range(K):
    cov_PP = covs[k][np.ix_(past_mask, past_mask)]
    mu_P = means[k, past_mask]

    log_pdf = multivariate_normal(mu_P, cov_PP, allow_singular=True).logpdf(x_P)
    log_contribs[k] = np.log(props[k]) + log_pdf
    

log_normal_const = logsumexp(log_contribs)
posteriors = np.exp(log_contribs - log_normal_const)

In [438]:
# conditional expectation

for k in range(K):
    cov_PP = covs[k][np.ix_(past_mask, past_mask)]
    cov_FP = covs[k][np.ix_(future_mask, past_mask)]

    mu_P = means[k, past_mask]
    mu_F = means[k, future_mask]

    residual = x_P - mu_P
    adjusted_weights = np.linalg.solve(cov_PP, residual)
    
    y_ik[k, :] = mu_F + cov_FP @ adjusted_weights

y_hat = np.dot(posteriors, y_ik)
forecasted_features = y_hat.reshape(h, len(spectral_indices))

In [439]:
#rescale
dummy_matrix = np.zeros((d-h, X.shape[1])) 
dummy_matrix[:, responses] = forecasted_features
unscaled_dummy = scaler.inverse_transform(dummy_matrix)
final_predictions = unscaled_dummy[:, responses]
df_pred = pd.DataFrame(final_predictions, columns=spectral_indices)
df_pred

,NDVI_mean,NDRE_mean,NDWI_mean
0,0.235233,0.117748,-0.067476
1,0.237648,0.115777,-0.064448
2,0.235314,0.109853,-0.067461
3,0.231734,0.104509,-0.073172
4,0.225847,0.098011,-0.081875
5,0.218692,0.091001,-0.091649


In [440]:
# Compare
X_test_raw.iloc[:,responses].tail(6)

,NDVI_mean,NDRE_mean,NDWI_mean
2119,0.238500,0.146078,-0.103053
2120,0.243403,0.150845,-0.104807
2121,0.247493,0.154405,-0.105438
2122,0.250837,0.156910,-0.105017
2123,0.254161,0.159280,-0.103811
2124,0.257459,0.161526,-0.101908


# References

Eirola, E. and Lendasse, A. (2013). Gaussian Mixture Models for Time Series Modelling, Forecasting, and Interpolation. [online] Available at: https://research.cs.aalto.fi/aml/Publications/Publication204.pdf